In [ ]:
# %pip install nlpaug

import pandas as pd
import nltk
import nlpaug.augmenter.word as naw
import numpy as np

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')

# Some constants
dataset_path = "../resources/dataset"
dataset = 'youtoxic_english_1000.csv'
augmented_dataset = f"synonym_{dataset}"
original_dataset = f"{dataset_path}/{dataset}"

# Array of columns to use for augmenting dataset
targets = [
    'IsToxic', 
    'IsAbusive', 
    'IsProvocative', 
    'IsObscene', 
    'IsHatespeech', 
    'IsRacist',
    'IsThreat',
    'IsReligiousHate',
    'IsNationalist'
]

# Load original dataset
df = pd.read_csv(original_dataset)
df_work = df.drop_duplicates(subset=['Text'])

def augment_text_data(df, text_column='Text', output_categories=None, num_augments=3):
    """
    Augments the text data in a DataFrame using Synonym Replacement.

    Args:
        df (pd.DataFrame): The original DataFrame.
        text_column (str): The name of the column containing the text.
        output_categories (list): List of columns to keep the original labels.
        num_augments (int): Number of augmented texts to generate per original text.

    Returns:
        pd.DataFrame: A new DataFrame containing the original and augmented data.
    """
    if output_categories is None:
        # Default to the categories you provided if not specified
        output_categories = [
            'IsToxic', 'IsAbusive', 'IsProvocative', 'IsObscene', 
            'IsHatespeech', 'IsRacist', 'IsThreat', 'IsReligiousHate', 
            'IsNationalist'
        ]

    # 1. Initialize the Synonym Replacement Augmenter
    # 'aug_word_p' is the percentage of words to be augmented (e.g., 0.3 means 30%)
    aug = naw.SynonymAug(aug_min=1, aug_p=0.3) 

    augmented_data = []
    positive_count = 0

    print(f"Starting augmentation on {len(df)} records...")
    
    # 2. Iterate through the DataFrame rows
    for index, row in df.iterrows():
        # Check if one of the selected categories is true
        is_positive = row[output_categories].sum() > 0
        
        if is_positive:
            positive_count += 1
            original_text = row[text_column]
            
            # Keep the original record
            original_record = row.copy()
            augmented_data.append(original_record)

            # 3. Generate the specified number of augmented texts
            for i in range(num_augments):
                # The augment() method takes a string and returns a list of augmented strings
                augmented_texts = aug.augment(original_text, n=1)
                
                if augmented_texts:
                    augmented_text = augmented_texts[0]
                    
                    # Create a new record with the augmented text
                    new_row = row.copy()
                    new_row[text_column] = augmented_text
                    
                    # The labels remain the same for the augmented text
                    augmented_data.append(new_row)

    # 4. Concatenate all data into a final DataFrame
    final_df = pd.DataFrame(augmented_data)
    
    print("-" * 50)
    print(f"Original Records: {len(df)}")
    print(f"Positive Records Augmented: {positive_count}")
    print(f"New Augmented Records Added: {positive_count * num_augments}")
    print(f"Final Dataset Size: {len(final_df)} records.")
    print("-" * 50)
    
    return final_df

augmented_df = augment_text_data(
    df_work, 
    text_column='Text', 
    output_categories=targets, 
    num_augments=4
)

print(f"\nTotal Records: {len(augmented_df)}")
# Save new dataset
augmented_df.to_csv(f"{dataset_path}/{augmented_dataset}")

[nltk_data] Downloading package wordnet to /home/vscode/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/vscode/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/vscode/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


Starting augmentation on 997 records...
Augmentation complete. Final dataset size: 4985 records.

Total Records: 4985
